# Imports

In [1]:
import pandas as pd
import os

In [2]:
df = pd.read_csv("data/meta_donnees.csv")

/var/folders/jh/0dqbxhtd4x3_mzgg04stlzg80000gn/T/ipykernel_63103/3882486439.py:1: DtypeWarning: Columns (0: departement-nom, 1: departement-insee, 2: identifiant de circonscription, 3: pdf, 4: suppleant-nom, 5: suppleant-prenom, 6: suppleant-sexe, 7: suppleant-age, 8: suppleant-age-calcule, 9: suppleant-age-tranche, 10: suppleant-profession, 11: suppleant-mandat-en-cours, 12: suppleant-mandat-passe, 13: suppleant-associations, 14: suppleant-autres-statuts, 15: suppleant-soutien, 16: suppleant-liste, 17: suppleant-decorations) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/meta_donnees.csv")


# Data Processing

## Etude des données

In [3]:
df.columns

Index(['id', 'date', 'subject', 'title', 'contexte-election', 'contexte-tour',
       'cote', 'departement', 'departement-nom', 'departement-insee',
       'identifiant de circonscription', 'images', 'pdf', 'ocr_url',
       'titulaire-nom', 'titulaire-prenom', 'titulaire-sexe', 'titulaire-age',
       'titulaire-age-calcule', 'titulaire-age-tranche',
       'titulaire-profession', 'titulaire-mandat-en-cours',
       'titulaire-mandat-passe', 'titulaire-associations',
       'titulaire-autres-statuts', 'titulaire-soutien', 'titulaire-liste',
       'titulaire-decorations', 'suppleant-nom', 'suppleant-prenom',
       'suppleant-sexe', 'suppleant-age', 'suppleant-age-calcule',
       'suppleant-age-tranche', 'suppleant-profession',
       'suppleant-mandat-en-cours', 'suppleant-mandat-passe',
       'suppleant-associations', 'suppleant-autres-statuts',
       'suppleant-soutien', 'suppleant-liste', 'suppleant-decorations'],
      dtype='str')

In [4]:
df.shape

(33030, 42)

In [5]:
df.head()

,id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,...,suppleant-age-calcule,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations
0,EL009_L_1958_11_001_01_1_PF_01,1958-11-23,France;Élections législatives;Assemblée Nation...,"Élections législatives de 1958, Ain - 01, circ...",législatives,1,EL009,01,Ain,01 - Ain,...,non mentionné,non mentionné,cultivateur,maire;conseiller général,non mentionné,non mentionné,non mentionné,Parti radical,non mentionné,non
1,EL009_L_1958_11_001_01_1_PF_02,1958-11-23,France;Ve République;Élections législatives;As...,"Élections législatives de 1958, Ain - 01, circ...",législatives,1,EL009,01,Ain,01 - Ain,...,non mentionné,non mentionné,cultivateur,conseiller municipal,non mentionné,non mentionné,prisonnier de guerre,Union pour la nouvelle République,non mentionné,non
2,EL009_L_1958_11_001_01_1_PF_03,1958-11-23,Élections législatives;France;Assemblée Nation...,"Élections législatives de 1958, Ain - 01, circ...",législatives,1,EL009,01,Ain,01 - Ain,...,non mentionné,non mentionné,cultivateur,non mentionné,non mentionné,non mentionné,non mentionné,Parti communiste français,non mentionné,non
3,EL009_L_1958_11_001_01_1_PF_04,1958-11-23,Élections législatives;France;Assemblée Nation...,"Élections législatives de 1958, Ain - 01, circ...",législatives,1,EL009,01,Ain,01 - Ain,...,35,entre 30 et 39 ans,greffier de paix,conseiller municipal;conseiller général,non mentionné,non mentionné,combattant,non mentionné,non mentionné,oui
4,EL009_L_1958_11_001_01_1_PF_05,1958-11-23,Ve République;Assemblée Nationale;Élections lé...,"Élections législatives de 1958, Ain - 01, circ...",législatives,1,EL009,01,Ain,01 - Ain,...,non mentionné,non mentionné,cultivateur;président Coopérative élevage,non mentionné,non mentionné,non mentionné,non mentionné,Centre national des indépendants et paysans,non mentionné,non


In [6]:
df["titulaire-liste"].unique()

<StringArray>
[                                                                                                                     'non mentionné',
                                                                                                        'Action familiale et sociale',
                                                                                     'Union des républicains radicaux et socialistes',
                                                                                            'Union républicaine sociale et familiale',
                                                                                                                              'Union',
                                                                                                             'Rassemblement national',
 'Union et concorde pour la défense de l'unité française dans l'Europe des intérêts nationaux et de l'ensemble de la circonscription',
                                         

In [7]:
colonne = pd.DataFrame(df["titulaire-liste"].unique(), columns=["titulaire-liste"])
colonne.to_csv("titulaire_liste.csv", index=False)

## Merging des textes

In [8]:
def load_text_multi(id_value, folder_paths):
    if pd.isna(id_value):
        return None

    id_str = str(id_value).strip()

    for folder in folder_paths:
        file_path = os.path.join(folder, f"{id_str}.txt")
        if os.path.exists(file_path):
            with open(file_path, "r", encoding="utf-8") as f:
                return f.read()

    return None

In [9]:
folders = [
    "data/textes_73",
    "data/textes_78",
    "data/textes_81",
    "data/textes_88",
    "data/textes_93"]

df["texte"] = df["id"].apply(lambda x: load_text_multi(x, folders))

df = df[df["texte"].notna()].copy()

In [10]:
df.shape

(21167, 43)

In [11]:
df = df.rename(columns={"titulaire-liste": "parti"})
df = df[["parti", "texte"]]

In [12]:
df.info()

<class 'pandas.DataFrame'>
Index: 21167 entries, 10775 to 32741
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   parti   21167 non-null  str  
 1   texte   21167 non-null  str  
dtypes: str(2)
memory usage: 496.1 KB


In [13]:
print(df[df["parti"]=="non mentionné"].count())

parti    8691
texte    8691
dtype: int64


## Mapping des partis

In [14]:
df_map = pd.read_csv("data/mapping_partis.csv")

In [15]:
df_map["orientation"].value_counts(dropna=False)

orientation
a_verifier    1287
gauche         669
centre         266
droite         251
inconnu          1
Name: count, dtype: int64

In [16]:
mapping_dict = dict(zip(
    df_map["titulaire-liste"],
    df_map["orientation"]))

df["orientation"] = df["parti"].map(mapping_dict)

In [17]:
df["orientation"].value_counts(dropna=False)

orientation
inconnu       8691
a_verifier    6275
gauche        3601
droite        1601
centre         999
Name: count, dtype: int64

In [24]:
df = df[df["orientation"].isin(["droite", "gauche", "centre"])]
df = df[["texte", "orientation"]]
df["orientation"].value_counts(dropna=False)

orientation
gauche    3601
droite    1601
centre     999
Name: count, dtype: int64

In [25]:
df = df.reset_index(drop=True)
df.to_csv("data/df_final.csv")